In [1]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, models

# ✅ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load Model
model = models.efficientnet_b0(weights=None)

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 2)
)

model.load_state_dict(torch.load(r"C:\Users\vansh\OneDrive\Desktop\Deepfake\experiments\AI_models\video\efficientnet_video_pytorch.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Class names
class_names = ['fake', 'real']

# ✅ Transform (IMPORTANT)
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# 🎥 Extract frames
def extract_frames(video_path, frame_skip=5):
    cap = cv2.VideoCapture(video_path)
    frames = []
    count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if count % frame_skip == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = transform(frame)
            frames.append(frame)

        count += 1

    cap.release()
    return torch.stack(frames) if len(frames) > 0 else None

# 🔮 Predict video
def predict_video(video_path):
    frames = extract_frames(video_path)

    if frames is None:
        print("❌ No frames extracted.")
        return

    frames = frames.to(device)

    with torch.no_grad():
        outputs = model(frames)
        probs = torch.softmax(outputs, dim=1)

    avg_prediction = torch.mean(probs, dim=0)

    predicted_class = class_names[torch.argmax(avg_prediction).item()]
    confidence = torch.max(avg_prediction).item()

    print("🎯 Final Video Prediction:", predicted_class)
    print("🔥 Confidence:", round(confidence, 4))
  

# ✅ Test video
video_path = r"D:\Screenshots\Random\VID-20260325-WA0005.mp4"

predict_video(video_path)

🎯 Final Video Prediction: real
🔥 Confidence: 0.6726
